# Formation DTX

[Lien du cours](https://github.com/asardell/data-science-python/blob/main/1-Introduction-data-science.md#chapitre-1--introduction-%C3%A0-la-data-science)

Ce notebook porte sur le sujet des DPE <br>
Lien de l'API : https://data.ademe.fr/datasets/dpe03existant

## Collecte des données

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

### Via l'url en csv



URL pour télécharger les données : https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines?qs=code_departement_ban%3A%2269%22+AND+surface_habitable_logement%3A%3C150&format=csv&size=10000



In [ ]:
#df = pd.read_csv("/content/dpe03existant.csv")

### Via l'API avec Python

In [ ]:
import requests

url = "https://data.ademe.fr/data-fair/api/v1/datasets/meg-83tjwtg8dyz4vv7h1dqe/lines"

params = {
    "qs": "code_departement_ban:69",
    "size": 500
}

toutes_les_lignes = []

while url:

    response = requests.get(url, params=params)
    data = response.json()

    # Ajouter les lignes
    toutes_les_lignes.extend(data["results"])

    print(f"Nombre de lignes : {len(toutes_les_lignes)}")

    # Pour les appels suivants, on utilise directement next
    url = data.get("next")

    # Les paramètres sont déjà présents dans next
    params = None

    if len(toutes_les_lignes) >= 10000:
        break

df = pd.DataFrame(toutes_les_lignes)

print(df.shape)

### Exporter en CSV

Pratique si votre session google colab devient inactive, pensez à télécharger le csv pour éviter de le générer à chaque fois qu'on run tout le notebook

In [ ]:
df.to_csv("dpe03existant.csv")

## Statistiques descriptives

In [ ]:
#afficher un extrait du dataframe
df.head(3)

In [ ]:
# taille du dataframe
df.shape

In [ ]:
# lister les colonnes
sorted(df.columns.to_list())

In [ ]:
# nombre de modalités unique
df["etiquette_dpe"].unique()

In [ ]:
# répartition en effectif
df["etiquette_dpe"].value_counts()

In [ ]:
# répartition en %
df["type_energie_n1"].value_counts(normalize=True)

In [ ]:
# indicateur de position et dispersion
print(df["surface_habitable_logement"].mean())
print(df["surface_habitable_logement"].min())
print(df["surface_habitable_logement"].max())
print(df["surface_habitable_logement"].median())
print(df["surface_habitable_logement"].var())
print(df["surface_habitable_logement"].std())

In [ ]:
# calculer des déciles
df["surface_habitable_logement"].describe(percentiles=[0.1,0.2,
                                                       0.3,0.4,0.5,
                                                       0.6,0.7,0.8,
                                                       0.9])

In [ ]:
# calcul des centiles
import numpy as np
centiles = np.arange(0.01, 1.00, 0.01)
df["surface_habitable_logement"].describe(percentiles=centiles)

In [ ]:
# calcul sur tout le df
df.describe().round(1)

## Graphiques

### Histogramme

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(df['surface_habitable_logement'], bins = 10)
plt.title("Histogramme de la surface_habitable_logement")
plt.show()

### Boxplot

![Image](https://miro.medium.com/max/9000/1*2c21SkzJMf3frPXPAR_gZA.png)

In [ ]:
# avec seaborn
sns.boxplot(y=df['surface_habitable_logement'], color='lightgreen')
plt.title("Boxplot surface_habitable_logement")
plt.show()

In [ ]:
# avec plotly
import plotly.express as px
fig = px.box(df, y='surface_habitable_logement',
             title="surface_habitable_logement")
fig.show()

print(df["surface_habitable_logement"].describe())

In [ ]:
# boxplot bivarié
fig = px.box(df, x='etiquette_dpe',
             y='deperditions_murs', title="deperditions_murs par étiquette DPE")
fig.show()
# penser à zoomer avec l'outil plotly

### Diagramme

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ordre_dpe = ["A", "B", "C", "D", "E", "F", "G"]

couleurs_dpe = {
    "A": "#009900",
    "B": "#66CC33",
    "C": "#CCFF00",
    "D": "#FFFF00",
    "E": "#FFCC00",
    "F": "#FF9900",
    "G": "#FF0000"
}

sns.countplot(
    data=df,
    x="etiquette_dpe",
    order=ordre_dpe,
    hue="etiquette_dpe",
    hue_order=ordre_dpe,
    palette=couleurs_dpe,
    legend=False
)

plt.xlabel("Étiquette DPE")
plt.ylabel("Nombre")
plt.title("Répartition des étiquettes DPE")
plt.xticks(rotation=0)

plt.show()

### Diagramme bivarié

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

table = pd.crosstab(
    df["type_energie_n1"],
    df["etiquette_dpe"],
    normalize="index"
) * 100

# Ordre des étiquettes
ordre_dpe = ["A", "B", "C", "D", "E", "F", "G"]

table = table.reindex(columns=ordre_dpe)

# Palette DPE
couleurs = plt.cm.RdYlGn_r(
    [0, 1/6, 2/6, 3/6, 4/6, 5/6, 1]
)

ax = table.plot(
    kind="barh",
    stacked=True,
    figsize=(14, 20),
    color=couleurs
)

ax.set_xlabel("Pourcentage (%)")
ax.set_ylabel("Type d'énergie")
ax.set_title("Étiquette DPE selon le type d'énergie")

ax.legend(
    title="Étiquette DPE",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

### Nuage de points

In [ ]:
import plotly.express as px

couleurs_dpe = {
    "A": "#009900",
    "B": "#66CC33",
    "C": "#CCFF00",
    "D": "#FFFF00",
    "E": "#FFCC00",
    "F": "#FF9900",
    "G": "#FF0000"
}

fig = px.scatter(
    df,
    x="deperditions_murs",
    y="besoin_chauffage",
    color="etiquette_dpe",
    color_discrete_map=couleurs_dpe,
    category_orders={"etiquette_dpe": ["A", "B", "C", "D", "E", "F", "G"]},
    title="Déperditions murs vs besoin chauffage"
)

fig.show()

### Heatmap de corrélation

In [ ]:
ls_columns = [
    'besoin_chauffage',
    'surface_habitable_logement',
    'deperditions_enveloppe',
    'deperditions_ponts_thermiques',
    'deperditions_murs',
    'deperditions_planchers_hauts',
    'deperditions_planchers_bas',
    'deperditions_portes',
    'deperditions_baies_vitrees',
    'deperditions_renouvellement_air'
    ]
corr = df[ls_columns].corr().round(1)
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Heatmap des corrélations")
plt.show()

In [ ]:
import plotly.figure_factory as ff

corr = df[ls_columns].corr().round(1).values
fig = ff.create_annotated_heatmap(corr, x=ls_columns, y=ls_columns, colorscale='RdBu', showscale=True)
fig.show()

## Pandas profiling HTML

il faut installer la librairie qui n'est pas présente par défaut

In [ ]:
pip install ydata_profiling

In [ ]:
from ydata_profiling import ProfileReport

ls_columns = [
 'etiquette_dpe',
 'etiquette_ges',
 'type_batiment',
 'annee_construction',
 'periode_construction',
 'type_installation_chauffage',
 'type_installation_ecs',
 'hauteur_sous_plafond',
 'nombre_appartement',
 'nombre_niveau_immeuble',
 'nombre_niveau_logement',
 'typologie_logement',
 'surface_habitable_logement',
 'classe_inertie_batiment',
 'classe_altitude',
 'zone_climatique',
 'indicateur_confort_ete',
 'protection_solaire_exterieure',
 'logement_traversant',
 'presence_brasseur_air',
 'inertie_lourde',
 'isolation_toiture',
 'deperditions_enveloppe',
 'deperditions_ponts_thermiques',
 'deperditions_murs',
 'deperditions_planchers_hauts',
 'deperditions_planchers_bas',
 'deperditions_portes',
 'deperditions_baies_vitrees',
 'deperditions_renouvellement_air',
 'conso_5_usages_ep',
 'conso_chauffage_ep',
 'conso_ecs_ep',
 'conso_refroidissement_ep',
 'conso_eclairage_ep',
 'conso_auxiliaires_ep',
 'type_energie_n1',
 'cout_total_5_usages',
 'cout_chauffage',
 'cout_ecs',
 'cout_refroidissement',
 'cout_eclairage',
 'cout_auxiliaires',
 'type_energie_principale_chauffage',
 'type_generateur_chauffage_principal',
 'type_installation_chauffage_n1'
 ]

profile = ProfileReport(df[ls_columns],
                        title="Profiling ADEME", explorative=True)


# export en HTML
profile.to_file("profiling_ademe.html")

## Machine Learning

### Classification

#### Choix des colonnes

In [ ]:
# choix des variables explicatives + cible
ls_columns = [
 'etiquette_dpe',
 'periode_construction',
 'hauteur_sous_plafond',
 'surface_habitable_logement',
 'deperditions_enveloppe',
 'deperditions_ponts_thermiques',
 'deperditions_murs',
 'deperditions_planchers_hauts',
 'deperditions_planchers_bas',
 'deperditions_portes',
 'deperditions_baies_vitrees',
 'deperditions_renouvellement_air',
 'type_energie_n1',
 ]

df1 = df[ls_columns]

In [ ]:
df1.shape

#### Gestion des valeurs manquantes

In [ ]:
# gestion des valeurs manquantes
missing_rate = df1.isna().mean().sort_values(ascending=False) * 100
print("Taux de valeurs manquantes (%) par colonne :")
print(missing_rate)

In [ ]:
# remplacement par la médiane ou la modalité "Inconnue"
num_cols = df1.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df1.select_dtypes(include=["object"]).columns

for col in num_cols:
    median_val = df1[col].median()
    df1[col] = df1[col].fillna(median_val)

for col in cat_cols:
    mode_val = df1[col].mode(dropna=True)
    if len(mode_val) > 0:
        df1[col] = df1[col].fillna(mode_val.iloc[0])
    else:
        df1[col] = df1[col].fillna("Inconnu")

print("Imputation terminée.")
print(df1.isna().sum())

#### Création variable cible à prédire

In [ ]:
# Création de la variable cible
df1["passoire_energetique"] = df1["etiquette_dpe"].isin(["E", "F", "G"])

#### Séparation de X et Y

In [ ]:
Y = df1["passoire_energetique"]
X = df1.drop(columns=["passoire_energetique", "etiquette_dpe"])

#### Binarisation des variables catégorielles

avec `drop_first=True` pour avoir n-1 colonnes car on peut déduire la dernière (ex: homme/femme, une colonne sufit)

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
# Vérification
X.head(5)

#### Gestion des valeurs extrêmes par clipping

In [ ]:
X.select_dtypes(include=["int64", "float64"])

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

def manage_outlier(col, low=0.01, high=0.99):
    q_low = col.quantile(low)
    q_high = col.quantile(high)
    return col.clip(lower=q_low, upper=q_high)

for c in num_cols:
    X[c] = manage_outlier(X[c])

print("Outliers capés")
X.describe(percentiles=[0.01,0.99])

#### Echantillonnage pour séparer le train et le test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

print("Train/test split réalisé.")

In [ ]:
print("Taille totale du dataset :", len(X))
print("Taille X_train :", len(X_train))
print("Taille X_test  :", len(X_test))
print("Taille y_train :", len(y_train))
print("Taille y_test  :", len(y_test))

In [ ]:
print("\nDistribution de y_train :")
print(y_train.value_counts(normalize=True))

print("\nDistribution de y_test :")
print(y_test.value_counts(normalize=True))

#### Paramétrage de l'arbre

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_arbre = DecisionTreeClassifier(max_depth=4, min_samples_leaf=300, min_samples_split=500, random_state=42)
model_arbre.fit(X_train, y_train)

In [ ]:
type(model_arbre)

#### Interprétation de l'arbre

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(16,10))
plot_tree(model_arbre, feature_names=X.columns, filled=True, fontsize=10)
plt.show()

#### Prédiction sur y_test

In [ ]:
# Prédiction
y_pred = model_arbre.predict(X_test)
y_pred_proba = model_arbre.predict_proba(X_test)

In [ ]:
y_pred[0:15]

In [ ]:
y_pred_proba[0:7]

#### Evaluation du modèle

In [ ]:
y_test.value_counts()

In [ ]:
from sklearn.metrics import confusion_matrix

# Matrice de confusion
mc = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    columns=['pred_pas_passoire', 'pred_passoire'],
    index=['obs_pas_passoire', 'obs_passoire']
)

# Ajouter les totaux lignes et colonnes
mc['Total'] = mc.sum(axis=1)
mc.loc['Total'] = mc.sum(axis=0)

mc

![Image](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRSzdNY1FQXuexzZbNVucPhzA2wxAovu971wp7FWef1xdl48W1ZLe0Jfv0p&s=10)

In [ ]:
# Accuracy
from sklearn.metrics import accuracy_score
print('Accuracy :', accuracy_score(y_test, y_pred))

In [ ]:
# Rappel
from sklearn.metrics import recall_score
print('Recall (binaire) :', recall_score(y_test, y_pred, average='binary', pos_label=1))


In [ ]:
# Precision
from sklearn.metrics import precision_score
print('Precision (binaire) :', precision_score(y_test, y_pred, average='binary', pos_label=1))

In [ ]:
# F1-score
from sklearn.metrics import f1_score
print('F1-score (binaire) :', f1_score(y_test, y_pred, average='binary', pos_label=1))

In [ ]:
# Micro / Macro / Weighted
print('Recall (macro) :', recall_score(y_test, y_pred, average='macro'))
print('Precision (macro) :', precision_score(y_test, y_pred, average='macro'))
print('F1-score (macro) :', f1_score(y_test, y_pred, average='macro'))

print('Recall (weighted) :', recall_score(y_test, y_pred, average='weighted'))
print('Precision (weighted) :', precision_score(y_test, y_pred, average='weighted'))
print('F1-score (weighted) :', f1_score(y_test, y_pred, average='weighted'))

#### Arbre 2

In [ ]:
model_arbre2 = DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, min_samples_split=20, random_state=42)
model_arbre2.fit(X_train, y_train)

# Prédiction
y_pred2 = model_arbre2.predict(X_test)
y_pred_proba2 = model_arbre2.predict_proba(X_test)

# Métriques
print('Accuracy :', accuracy_score(y_test, y_pred2))
print('F1-score (macro) :', f1_score(y_test, y_pred2, average='macro'))

#### Courbe ROC

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Probabilités pour la classe positive
proba1 = y_pred_proba[:,1]
proba2 = y_pred_proba2[:,1]

# ROC
fpr1, tpr1, _ = roc_curve(y_test, proba1)
fpr2, tpr2, _ = roc_curve(y_test, proba2)

auc1 = roc_auc_score(y_test, proba1)
auc2 = roc_auc_score(y_test, proba2)

plt.figure(figsize=(8,6))
plt.plot(fpr1, tpr1, label=f'Modèle 1 (AUC={auc1:.2f})', color='blue')
plt.plot(fpr2, tpr2, label=f'Modèle 2 (AUC={auc2:.2f})', color='green')
plt.plot([0,1],[0,1],'k--', label='Aléatoire')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Comparaison des courbes ROC')
plt.legend()
plt.show()

#### Arbre avec GridSearch

In [ ]:
import numpy as np

parameters = {
    'max_depth': np.arange(3, 10, 2),
    'min_samples_leaf': np.arange(5, 250, 50),
    'min_samples_split': np.arange(10, 500, 50)
}

total_combinaisons = (
    len(parameters['max_depth']) *
    len(parameters['min_samples_leaf']) *
    len(parameters['min_samples_split'])
)

print(f"Nombre total de modèles à tester: {total_combinaisons}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

score = make_scorer(f1_score, pos_label=1)
model_arbre_grid = DecisionTreeClassifier()
grid_search = GridSearchCV(model_arbre_grid, parameters, scoring=score, verbose=2, cv = 5)
grid_search.fit(X_train, y_train)

In [ ]:
best_model = grid_search.best_estimator_

print("Meilleurs paramètres :", grid_search.best_params_)
print("Score f1 du meilleur modèle :", grid_search.best_score_)

In [ ]:
y_pred_grid = best_model.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_grid))

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Probabilités de prédiction pour la classe positive
y_proba_grid = best_model.predict_proba(X_test)[:,1]
y_proba_model1 = model_arbre.predict_proba(X_test)[:,1]
y_proba_model2 = model_arbre2.predict_proba(X_test)[:,1]

# Calcul ROC
fpr_grid, tpr_grid, _ = roc_curve(y_test, y_proba_grid)
fpr1, tpr1, _ = roc_curve(y_test, y_proba_model1)
fpr2, tpr2, _ = roc_curve(y_test, y_proba_model2)

# AUC
auc_grid = roc_auc_score(y_test, y_proba_grid)
auc1 = roc_auc_score(y_test, y_proba_model1)
auc2 = roc_auc_score(y_test, y_proba_model2)

# Tracé
plt.figure(figsize=(8,6))
plt.plot(fpr_grid, tpr_grid, label=f'GridSearchCV (AUC={auc_grid:.2f})', color='red')
plt.plot(fpr1, tpr1, label=f'Modèle 1 (AUC={auc1:.2f})', color='blue')
plt.plot(fpr2, tpr2, label=f'Modèle 2 (AUC={auc2:.2f})', color='green')
plt.plot([0,1],[0,1],'k--', label='Aléatoire')
plt.xlabel('Faux positifs (FPR)')
plt.ylabel('Vrais positifs (TPR)')
plt.title('Comparaison ROC entre modèles d\'arbre')
plt.legend(loc='lower right')
plt.show()

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Modèle initial
rf = RandomForestClassifier(random_state=42)

# Grille des hyperparamètres
param_grid = {
    'n_estimators': [5, 20, 50],
    'max_depth': [5, 10],
    'min_samples_split': [10, 50, 100],
    'min_samples_leaf': [5, 25, 50],
}

# Grid Search optimisant le F1 macro
grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=score,
    cv=5
)

grid_search_rf.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_search_rf.best_params_)

# Extraction du meilleur modèle
best_rf = grid_search_rf.best_estimator_

In [ ]:
y_pred_rf = best_rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

##### Feature importances

In [ ]:
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importances)

plt.figure(figsize=(10,6))
plt.bar(importances['Feature'], importances['Importance'])
plt.xticks(rotation=45)
plt.title("Feature Importances — Random Forest")
plt.show()

#### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Définition du modèle KNN
knn = KNeighborsClassifier()

# Paramètres à tester dans Grid Search
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance']
}

# Grid Search
grid_knn = GridSearchCV(knn, param_grid, scoring=score, cv=5, verbose=2)
grid_knn.fit(X_train, y_train)

# Meilleur modèle
best_knn = grid_knn.best_estimator_
print("Meilleur modèle KNN :", best_knn)
print("Meilleur score f1 (classe 0) :", grid_knn.best_score_)

# Prédictions sur l'ensemble de test
y_pred_knn = best_knn.predict(X_test)

print(classification_report(y_test, y_pred_knn))

### Régression

#### Préparation

In [ ]:
# choix des variables explicatives + cible
ls_columns = [
 'besoin_chauffage',
 'periode_construction',
 'hauteur_sous_plafond',
 'surface_habitable_logement',
 'deperditions_enveloppe',
 'deperditions_ponts_thermiques',
 'deperditions_murs',
 'deperditions_planchers_hauts',
 'deperditions_planchers_bas',
 'deperditions_portes',
 'deperditions_baies_vitrees',
 'deperditions_renouvellement_air',
 'type_energie_n1',
 ]

df2 = df[ls_columns]

#### Gestion des valeurs manquantes

In [ ]:
# remplacement par la médiane ou la modalité "Inconnue"
num_cols = df2.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df2.select_dtypes(include=["object"]).columns

for col in num_cols:
    median_val = df2[col].median()
    df2[col] = df2[col].fillna(median_val)

for col in cat_cols:
    mode_val = df2[col].mode(dropna=True)
    if len(mode_val) > 0:
        df2[col] = df2[col].fillna(mode_val.iloc[0])
    else:
        df2[col] = df2[col].fillna("Inconnu")

print("Imputation terminée.")
print(df2.isna().sum())

#### Analyse variable cible à prédire

In [ ]:
Y = df2["besoin_chauffage"]
X = df2.drop(columns=["besoin_chauffage"])

In [ ]:
import plotly.express as px

fig = px.box(
    df2,
    y="besoin_chauffage",
    title="Distribution du besoin de chauffage",
    labels={"besoin_chauffage": "Besoin de chauffage"}
)

fig.show()

#### Binarisation des variables catégorielles

In [ ]:
X = pd.get_dummies(X, drop_first=True)

#### Gestion des valeurs extrêmes par clipping

##### Pour X

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

def manage_outlier(col, low=0.001, high=0.999):
    q_low = col.quantile(low)
    q_high = col.quantile(high)
    return col.clip(lower=q_low, upper=q_high)

for c in num_cols:
    X[c] = manage_outlier(X[c])

print("Outliers capés")
X.describe(percentiles=[0.001,0.999])

##### Pour Y

In [ ]:
Y = manage_outlier(Y)

In [ ]:
import plotly.express as px

fig = px.box(
    Y,
    title="Distribution du besoin de chauffage",
    labels={"besoin_chauffage": "Besoin de chauffage"}
)

fig.show()

#### Echantillonnage pour séparer le train et le test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42
)

print("Train/test split réalisé.")

#### Entrainement du modèle de régression linéaire multiple

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

reg_lin = LinearRegression()
reg_lin.fit(X_train, y_train)

#### Interprétation des coefficients

In [ ]:
coeffs = pd.DataFrame(reg_lin.coef_, index=X_train.columns, columns=['Coefficient'])
intercept = reg_lin.intercept_

print("Coefficients :\n", coeffs.round(1))
print("Intercept :", intercept)

#### Normaliser et Standardiser les données

*Petit exemple sans conséquence sur la modélisation en cours*

In [ ]:
# Standardisation du besoin en chauffage
mean_besoin = df2['besoin_chauffage'].mean()
std_besoin = df2['besoin_chauffage'].std()
df2['besoin_chauffage_standardized'] = (df2['besoin_chauffage'] - mean_besoin) / std_besoin

df2[["besoin_chauffage","besoin_chauffage_standardized"]]

In [ ]:
print("Moyenne standardisée :", df2['besoin_chauffage_standardized'].mean().round(1))
print("Écart type standardisé :", df2['besoin_chauffage_standardized'].std())

##### X_train et X_test

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialiser le StandardScaler
scaler = StandardScaler()

# Ajuster (fit) le scaler uniquement sur les données d'entraînement et appliquer la transformation (fit_transform)
X_train_scaled = scaler.fit_transform(X_train)

# Appliquer la transformation sur les données de test (transform uniquement, sans fit)
X_test_scaled = scaler.transform(X_test)

##### y_train et y_test

In [ ]:
# Initialiser le StandardScaler pour les cibles (y)
scaler_y = StandardScaler()

# Convertir y_train et y_test en tableau NumPy pour appliquer reshape
y_train_scaled = scaler_y.fit_transform(np.array(y_train).reshape(-1, 1))
y_test_scaled = scaler_y.transform(np.array(y_test).reshape(-1, 1))

In [ ]:
reg_lin = LinearRegression()
reg_lin.fit(X_train_scaled, y_train_scaled)

coeffs = pd.DataFrame(reg_lin.coef_.T, index=X_train.columns, columns=['Coefficient'])
intercept = reg_lin.intercept_

print("Coefficients :\n", coeffs.round(2))
print("Intercept :", intercept)

#### Prédiction sur y_test

In [ ]:
y_pred_scaled = reg_lin.predict(X_test_scaled)
y_pred_scaled

##### Remettre à l'échelle la variable cible

In [ ]:
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))

pd.DataFrame(y_pred).round(1)

In [ ]:
y_pred = np.asarray(y_pred).ravel()
y_test = np.asarray(y_test).ravel()

print(y_test.shape)
print(y_pred.shape)

#### Evaluation du modèle

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE :", rmse)
print("MAE :", mae)
print("R² :", r2)

In [ ]:
import plotly.express as px
import numpy as np

fig = px.scatter(
    x=y_test,
    y=y_pred,
    labels={
        "x": "Valeurs observées",
        "y": "Valeurs prédites"
    },
    title="Régression linéaire : Observé vs Prévu",
    opacity=0.6
)

# Droite y = x
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

fig.add_scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode="lines",
    name="Prédiction parfaite",
    line=dict(color="red", dash="dash", width=2)
)

fig.update_layout(
    width=800,
    height=600
)

fig.show()

#### Arbre de régression

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Création de l'arbre
arbre_reg = DecisionTreeRegressor(max_depth=5, min_samples_split=50, min_samples_leaf=20, random_state=42)
arbre_reg.fit(X_train, y_train)

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,8))
plot_tree(arbre_reg, feature_names=X.columns, filled=True, fontsize=10)
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = arbre_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE :", mse)
print("RMSE :", rmse)
print("MAE :", mae)
print("R² :", r2)

### Clustering

#### Kmeans

*Petit exemple pédagogique*

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Création d'un dataset factice
# cluster_std=0.5  # peu de bruit
# cluster_std=1.0  # bruit modéré
# cluster_std=2.0  # beaucoup de bruit
# cluster_std=3.0  # beaucoup de recouvrement

X, _ = make_blobs(n_samples=200, centers=3, n_features=2, random_state=42, cluster_std=3.0)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

# Visualisation
plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis')
plt.scatter(centroids[:,0], centroids[:,1], color='red', marker='X', s=200)
plt.title("K-Means - exemple pédagogique")
plt.show()

##### Sans centrer et réduire

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import plotly.express as px

# Colonnes utilisées pour le clustering
cols = [
    'surface_habitable_logement',
    'conso_chauffage_ep'
]

df_clustering = df[cols].dropna().copy()

# KMeans
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(df_clustering)

df_clustering['cluster'] = labels

# Évaluation
score = silhouette_score(
    df_clustering[cols],
    labels
)

print("Silhouette score :", score)

# Centroïdes
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=cols
)

print("\nCentroïdes :")
print(centroids)

# ------------------------------------------------
# Graphique des clusters
# ------------------------------------------------

fig = px.scatter(
    df_clustering,
    x='surface_habitable_logement',
    y='conso_chauffage_ep',
    color='cluster',
    title='K-Means : clusters et centroïdes',
    labels={
        'surface_habitable_logement': 'Surface habitable',
        'conso_chauffage_ep': 'Consommation chauffage',
        'cluster': 'Cluster'
    },
    opacity=0.6
)

# Ajouter les centroïdes
fig.add_scatter(
    x=centroids['surface_habitable_logement'],
    y=centroids['conso_chauffage_ep'],
    mode='markers',
    name='Centroïdes',
    marker=dict(
        size=15,
        color='black',
        symbol='x'
    )
)

fig.update_layout(
    width=900,
    height=650
)

fig.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import pandas as pd
import plotly.express as px

# Colonnes utilisées pour le clustering
cols = [
    'surface_habitable_logement',
    'conso_chauffage_ep'
]

df_clustering = df[cols].dropna().copy()

# ------------------------------------------------
# Standardisation
# ------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(df_clustering[cols])

# ------------------------------------------------
# KMeans
# ------------------------------------------------

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(X)

df_clustering['cluster'] = labels

# ------------------------------------------------
# Évaluation
# ------------------------------------------------

score = silhouette_score(X, labels)

print("Silhouette score :", score)

# ------------------------------------------------
# Centroïdes standardisés
# ------------------------------------------------

centroids_scaled = kmeans.cluster_centers_

print("\nCentroïdes standardisés :")
print(
    pd.DataFrame(
        centroids_scaled,
        columns=cols
    )
)

# ------------------------------------------------
# Inverse transform des centroïdes
# ------------------------------------------------

centroids_original = scaler.inverse_transform(centroids_scaled)

centroids = pd.DataFrame(
    centroids_original,
    columns=cols
)

print("\nCentroïdes dans les unités originales :")
print(centroids)

# ------------------------------------------------
# Graphique
# ------------------------------------------------

fig = px.scatter(
    df_clustering,
    x='surface_habitable_logement',
    y='conso_chauffage_ep',
    color='cluster',
    title='K-Means : clusters et centroïdes',
    labels={
        'surface_habitable_logement': 'Surface habitable',
        'conso_chauffage_ep': 'Consommation chauffage',
        'cluster': 'Cluster'
    },
    opacity=0.6
)

# Ajouter les centroïdes dans les unités originales
fig.add_scatter(
    x=centroids['surface_habitable_logement'],
    y=centroids['conso_chauffage_ep'],
    mode='markers',
    name='Centroïdes',
    marker=dict(
        size=15,
        color='black',
        symbol='x'
    )
)

fig.update_layout(
    width=900,
    height=650
)

fig.show()

#### Classification Ascendante Hiérarchique

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Colonnes utilisées pour le clustering
cols = [
    'surface_habitable_logement',
    'conso_chauffage_ep',
    'hauteur_sous_plafond',
    'deperditions_murs',
]

df_cah = df[cols].dropna().copy()

# ------------------------------------------------
# Standardisation
# ------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(df_cah[cols])
Z = linkage(X, method='ward')

# Dendrogramme
plt.figure(figsize=(10,6))
dendrogram(Z, truncate_mode='level', p=7)
plt.title("Dendrogramme CAH - ADEME")
plt.show()

In [ ]:
# Attribution des clusters
df_cah['cluster_cah'] = fcluster(Z, t = 55, criterion='distance')

In [ ]:
df_cah['cluster_cah'].value_counts(normalize=True)

##### Analyse des groupes

In [ ]:
import plotly.express as px

# Les 4 variables à analyser
cols = [
    'surface_habitable_logement',
    'conso_chauffage_ep',
    'hauteur_sous_plafond',
    'deperditions_murs',
]
# Boucle sur les variables
for var in cols:

    fig = px.box(
        df_cah,
        x='cluster_cah',
        y=var,
        color='cluster_cah',
        title=f'Distribution de {var} selon les clusters CAH',
        labels={
            'cluster_cah': 'Cluster',
            var: var
        },
        points='outliers'
    )

    fig.update_layout(
        width=900,
        height=600
    )

    fig.show()

In [ ]:
medians = df_cah.groupby('cluster_cah')[cols].median().round(2)
medians

### Annexes

#### Text mining

In [ ]:
df["description_installation_chauffage_n1"]

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

# Charger les données
texts = df['description_installation_chauffage_n1'].astype(str)

# Nettoyage
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

texts_clean = texts.apply(clean_text)

# Tokenisation, suppression stopwords, lemmatisation
stop_words = set(stopwords.words('french'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

texts_processed = texts_clean.apply(preprocess)

In [ ]:
vectorizer = CountVectorizer()
X_counts = vectorizer.fit_transform(texts_processed)

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(texts_processed)

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

all_text = " ".join(texts_processed)
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)

plt.figure(figsize=(15,7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
from sklearn.cluster import KMeans

k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
clusters = kmeans.fit_predict(X_tfidf)
df['cluster'] = clusters

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(X_tfidf)

def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

display_topics(lda, tfidf_vectorizer.get_feature_names_out(), 10)

In [ ]:
for i in range(k):
    cluster_text = " ".join(df[df['cluster'] == i]['description_installation_chauffage_n1'].astype(str))
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(cluster_text)
    plt.figure(figsize=(12,5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Cluster {i}")
    plt.show()

#### Forecast

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fixer le seed pour reproductibilité
np.random.seed(42)

# Générer une série temporelle simulée
n_years = 10
dates = pd.date_range(start='2010-01-01', periods=n_years*12, freq='M')

# Tendance linéaire
trend = np.linspace(200, 250, n_years*12)

# Saisonnalité mensuelle (12 mois)
seasonal = 20 * np.sin(2 * np.pi * np.arange(n_years*12)/12)

# Bruit aléatoire
noise = np.random.normal(0, 5, n_years*12)

# Consommation totale
consommation = trend + seasonal + noise

# Créer DataFrame
df = pd.DataFrame({'date': dates, 'consommation': consommation})
df.set_index('date', inplace=True)

# Visualisation
df['consommation'].plot(figsize=(12,4), title='Consommation énergétique simulée')
plt.ylabel('kWh/m²')
plt.show()

In [ ]:
from prophet import Prophet

# Préparer le dataframe pour Prophet
df_prophet = df.reset_index().rename(columns={'date':'ds', 'consommation':'y'})

prophet_model = Prophet(yearly_seasonality=True)
prophet_model.fit(df_prophet)

future = prophet_model.make_future_dataframe(periods=12, freq='M')
forecast = prophet_model.predict(future)

prophet_model.plot(forecast)
plt.title('Prévision avec Prophet')
plt.show()

In [ ]:
from prophet import Prophet
import plotly.graph_objects as go

# Préparer le dataframe pour Prophet
df_prophet = df.reset_index().rename(
    columns={'date': 'ds', 'consommation': 'y'}
)

# Modèle Prophet
prophet_model = Prophet(yearly_seasonality=True)
prophet_model.fit(df_prophet)

# Prévisions sur 12 mois
future = prophet_model.make_future_dataframe(periods=12, freq='ME')
forecast = prophet_model.predict(future)

# Graphique Plotly
fig = go.Figure()

# Valeurs réelles
fig.add_trace(go.Scatter(
    x=df_prophet['ds'],
    y=df_prophet['y'],
    mode='lines',
    name='Consommation réelle'
))

# Prévision
fig.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat'],
    mode='lines',
    name='Prévision'
))

# Intervalle d'incertitude
fig.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat_upper'],
    mode='lines',
    line=dict(width=0),
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat_lower'],
    mode='lines',
    fill='tonexty',
    line=dict(width=0),
    name='Intervalle de confiance'
))

fig.update_layout(
    title='Prévision de la consommation avec Prophet',
    xaxis_title='Date',
    yaxis_title='Consommation',
    width=1000,
    height=600,
    hovermode='x unified'
)

fig.show()

####  Khi2 - Tester l'indépendance de 2 variables qualitatives

[plus d'info sur le cours STA](https://github.com/asardell/STA/blob/main/Chapitre_8.md)

In [ ]:
import requests

url = "https://data.ademe.fr/data-fair/api/v1/datasets/meg-83tjwtg8dyz4vv7h1dqe/lines"

params = {
    "qs": "code_departement_ban:69",
    "size": 500
}

toutes_les_lignes = []

while url:

    response = requests.get(url, params=params)
    data = response.json()

    # Ajouter les lignes
    toutes_les_lignes.extend(data["results"])

    print(f"Nombre de lignes : {len(toutes_les_lignes)}")

    # Pour les appels suivants, on utilise directement next
    url = data.get("next")

    # Les paramètres sont déjà présents dans next
    params = None

    if len(toutes_les_lignes) >= 1000:
        break

df = pd.DataFrame(toutes_les_lignes)

print(df.shape)

In [ ]:
# lister les colonnes
sorted(df.columns.to_list())

Poser $H_0$ et $H_1$.
- $H_0$ : Les variables sont indépendantes
- $H_1$ : Les variables ne sont pas indépendantes

Que peut-on affirmer avec un risque de se tromper $\alpha = 0.05$

In [ ]:
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency

obs = pd.crosstab(df["etiquette_dpe"],df["periode_construction"])

obs

In [ ]:
Khi2_obs, p_value, ddl, effectif_theorique = chi2_contingency(obs)

print(f'Khi2  : {Khi2_obs}')
print(f'p_value  : {p_value}')
print(f'effectif_theorique  : {effectif_theorique}')
print(f'ddl  : {ddl}')

p-value < 0.05 donc $H_1$ : Les variables ne sont pas indépendantes
J'ai 5% de chances de me tromper en affirmant qu'il y a un lien en ces deux variables
<br>
J'ai 1.0143300211072255e-84 de chances de me tromper en affirmant qu'il y a un lien en ces deux variables